In [7]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
from scipy import sparse
import joblib
import os

# ==============================
# 1. LOAD MULTI-DATASET LOGS
# ==============================

BASE_DATASET_PATH = "E:/LogAnomalyDetector/data/datasets"

# ==============================
# HDFS PATHS
# ==============================

hdfs_path = (
    f"{BASE_DATASET_PATH}/hdfs/HDFS_2k.log_structured.csv"
)

hdfs_label_path = (
    f"{BASE_DATASET_PATH}/hdfs/anomaly_label.csv"
)

# ==============================
# BGL PATHS
# ==============================

bgl_path = (
    f"{BASE_DATASET_PATH}/bgl/BGL_2k.log_structured.csv"
)

# ==============================
# LOAD DATASETS
# ==============================

hdfs_df = pd.read_csv(hdfs_path)

hdfs_labels = pd.read_csv(hdfs_label_path)

bgl_df = pd.read_csv(bgl_path)

print("HDFS Shape:", hdfs_df.shape)

print("HDFS Labels Shape:", hdfs_labels.shape)

print("BGL Shape:", bgl_df.shape)

# ==============================
# HDFS LABEL INSPECTION
# ==============================

print("\nHDFS Label Columns:")

print(
    hdfs_labels.columns.tolist()
)

print("\nHDFS Label Sample:")

print(
    hdfs_labels.head()
)

# ==============================
# 2. EXTRACT HDFS BLOCK IDS
# ==============================

def extract_block_id(text):

    text = str(text)

    match = re.search(
        r'(blk_-?\d+)',
        text
    )

    if match:

        return match.group(1)

    return np.nan

# ==============================
# DETECT MESSAGE COLUMN
# ==============================

possible_hdfs_message_cols = [
    "Content",
    "content",
    "message",
    "log",
    "EventTemplate"
]

hdfs_message_col = None

for col in possible_hdfs_message_cols:

    if col in hdfs_df.columns:

        hdfs_message_col = col

        break

if hdfs_message_col is None:

    raise ValueError(
        "No message column found in HDFS dataset"
    )

# ==============================
# EXTRACT BLOCK IDS
# ==============================

hdfs_df["BlockId"] = (
    hdfs_df[hdfs_message_col]
    .astype(str)
    .apply(extract_block_id)
)

print("\nExtracted Block IDs:")

print(
    hdfs_df["BlockId"]
    .head()
)

# ==============================
# 3. PREPARE HDFS LABELS
# ==============================

hdfs_labels.columns = [
    c.strip()
    for c in hdfs_labels.columns
]

possible_block_cols = [
    "BlockId",
    "Block ID",
    "blkId"
]

possible_label_cols = [
    "Label",
    "label"
]

block_col = None
label_col = None

for col in possible_block_cols:

    if col in hdfs_labels.columns:

        block_col = col

        break

for col in possible_label_cols:

    if col in hdfs_labels.columns:

        label_col = col

        break

if block_col is None:

    raise ValueError(
        "No BlockId column found in HDFS labels"
    )

if label_col is None:

    raise ValueError(
        "No label column found in HDFS labels"
    )

# ==============================
# NORMALIZE LABELS
# ==============================

hdfs_labels["true_label"] = (
    hdfs_labels[label_col]
    .astype(str)
    .str.lower()
    .apply(
        lambda x:
        1 if x in [
            "anomaly",
            "abnormal",
            "1"
        ]
        else 0
    )
)

# ==============================
# CREATE LABEL MAP
# ==============================

label_map = dict(
    zip(
        hdfs_labels[block_col],
        hdfs_labels["true_label"]
    )
)

# ==============================
# APPLY LABELS TO HDFS
# ==============================

hdfs_df["true_label"] = (
    hdfs_df["BlockId"]
    .map(label_map)
)

# Default missing labels to normal
hdfs_df["true_label"] = (
    hdfs_df["true_label"]
    .fillna(0)
    .astype(int)
)

print("\nHDFS Label Distribution:")

print(
    hdfs_df["true_label"]
    .value_counts()
)

# ==============================
# 4. ADAPTIVE NORMALIZATION
# ==============================

def normalize_hdfs(df):

    possible_message_cols = [
        "Content",
        "content",
        "message",
        "log",
        "EventTemplate"
    ]

    message_col = None

    for col in possible_message_cols:

        if col in df.columns:

            message_col = col

            break

    if message_col is None:

        raise ValueError(
            "No message column found in HDFS dataset"
        )

    normalized = pd.DataFrame()

    normalized["message"] = (
        df[message_col]
        .astype(str)
    )

    normalized["true_label"] = (
        df["true_label"]
        .astype(int)
    )

    normalized["source_dataset"] = "hdfs"

    return normalized


def normalize_bgl(df):

    possible_message_cols = [
        "Content",
        "content",
        "message",
        "log",
        "EventTemplate"
    ]

    message_col = None

    for col in possible_message_cols:

        if col in df.columns:

            message_col = col

            break

    if message_col is None:

        raise ValueError(
            "No message column found in BGL dataset"
        )

    possible_label_cols = [
        "Label",
        "label",
        "true_label"
    ]

    label_col = None

    for col in possible_label_cols:

        if col in df.columns:

            label_col = col

            break

    if label_col is None:

        raise ValueError(
            "No label column found in BGL dataset"
        )

    normalized = pd.DataFrame()

    normalized["message"] = (
        df[message_col]
        .astype(str)
    )

    normalized["true_label"] = (
        df[label_col]
        .astype(str)
        .str.lower()
        .apply(
            lambda x:
            1 if x in [
                "anomaly",
                "abnormal",
                "1",
                "-"
            ]
            else 0
        )
    )

    normalized["source_dataset"] = "bgl"

    return normalized

# ==============================
# NORMALIZE DATASETS
# ==============================

hdfs_normalized = normalize_hdfs(hdfs_df)

bgl_normalized = normalize_bgl(bgl_df)

print("\nHDFS Normalized Shape:")

print(hdfs_normalized.shape)

print("\nBGL Normalized Shape:")

print(bgl_normalized.shape)

# ==============================
# 5. COMBINE DATASETS
# ==============================

df = pd.concat(
    [
        hdfs_normalized,
        bgl_normalized
    ],
    ignore_index=True
)

print("\nCombined Dataset Shape:")

print(df.shape)

print("\nDataset Distribution:")

print(
    df["source_dataset"]
    .value_counts()
)

# ==============================
# 6. BASIC CLEANING
# ==============================

df = df.dropna(
    subset=['message']
)

df['message'] = (
    df['message']
    .astype(str)
    .str.lower()
    .str.strip()
)

# ==============================
# REMOVE DATASET IDENTITY TOKENS
# ==============================

def clean_dataset_tokens(text):

    text = str(text)

    text = re.sub(
        r'blk_-?\d+',
        'BLOCK_ID',
        text
    )

    text = re.sub(
        r'0x[a-fA-F0-9]+',
        'HEX_ID',
        text
    )

    text = re.sub(
        r'\b\d+\b',
        'NUM',
        text
    )

    text = re.sub(
        r'node-\d+',
        'NODE_ID',
        text
    )

    text = re.sub(
        r'\b[a-f0-9]{8,}\b',
        'HASH_ID',
        text
    )

    return text

df["message"] = (
    df["message"]
    .apply(clean_dataset_tokens)
)

print("\nAfter Cleaning:")

print(df.shape)

# ==============================
# 7. REMOVE VECTOR-LIKE
#    CORRUPTED ROWS
# ==============================

def is_vector_like(message):

    message = str(message)

    if len(message) == 0:

        return False

    comma_ratio = (
        message.count(",")
        / max(len(message), 1)
    )

    digit_ratio = (
        sum(c.isdigit() for c in message)
        / max(len(message), 1)
    )

    return (
        comma_ratio > 0.30
        and digit_ratio > 0.50
    )

vector_mask = (
    df["message"]
    .apply(is_vector_like)
)

vector_rows = df[vector_mask]

print(
    "\nDetected Vector-Like Rows:",
    len(vector_rows)
)

df = df[~vector_mask]

print(
    "\nDataset Shape After Filtering:",
    df.shape
)

# ==============================
# 8. LABEL CHECK
# ==============================

df['true_label'] = (
    df['true_label']
    .astype(int)
)

print("\nLabel Distribution:")

print(
    df['true_label']
    .value_counts()
)

print("\nDataset-wise Labels:")

print(
    pd.crosstab(
        df["source_dataset"],
        df["true_label"]
    )
)

# ==============================
# 9. PREPARE DATA
# ==============================

X_text = df['message']

y = df['true_label'].values

# ==============================
# 10. TF-IDF FEATURES
# ==============================

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=4000,
    min_df=2,
    max_df=0.95
)

X_tfidf = (
    vectorizer.fit_transform(X_text)
)

print("\nTF-IDF Shape:")

print(X_tfidf.shape)

# ==============================
# 11. ADVANCED STRUCTURED FEATURES
# ==============================

def extract_structured_features(text):

    text = str(text)

    tokens = text.split()

    message_length = len(text)

    token_count = len(tokens)

    digit_ratio = (
        sum(c.isdigit() for c in text)
        / max(message_length, 1)
    )

    uppercase_ratio = (
        sum(c.isupper() for c in text)
        / max(message_length, 1)
    )

    special_char_count = len(
        re.findall(
            r'[^a-zA-Z0-9\s]',
            text
        )
    )

    numeric_token_count = sum(
        token.isdigit()
        for token in tokens
    )

    unique_token_ratio = (
        len(set(tokens))
        / max(token_count, 1)
    )

    contains_ip = int(bool(
        re.search(
            r'\b(?:\d{1,3}\.){3}\d{1,3}\b',
            text
        )
    ))

    contains_url = int(bool(
        re.search(
            r'https?://|www\.',
            text
        )
    ))

    contains_path = int(bool(
        re.search(
            r'[a-zA-Z]:\\\\|/var/|/home/|/tmp/',
            text
        )
    ))

    contains_port = int(bool(
        re.search(
            r'port\s+\d+',
            text
        )
    ))

    contains_hex = int(bool(
        re.search(
            r'hex_id',
            text.lower()
        )
    ))

    # ==========================
    # ERROR FEATURES
    # ==========================

    error_keywords = [
        "error",
        "fail",
        "failed",
        "fatal",
        "critical",
        "warning",
        "timeout",
        "denied",
        "exception",
        "panic"
    ]

    error_keyword_count = sum(
        keyword in text.lower()
        for keyword in error_keywords
    )

    contains_error = int(
        "error" in text.lower()
    )

    contains_warning = int(
        "warning" in text.lower()
    )

    contains_timeout = int(
        "timeout" in text.lower()
    )

    contains_failure = int(
        "failure" in text.lower()
    )

    contains_exception = int(
        "exception" in text.lower()
    )

    # ==========================
    # ENTROPY-LIKE FEATURES
    # ==========================

    repeated_token_ratio = (
        1 -
        (
            len(set(tokens))
            / max(len(tokens), 1)
        )
    )

    average_token_length = np.mean(
        [len(t) for t in tokens]
    ) if len(tokens) > 0 else 0

    # ==========================
    # FEATURE STABILIZATION
    # ==========================

    message_length = min(
        message_length,
        10000
    )

    special_char_count = min(
        special_char_count,
        5000
    )

    token_count = min(
        token_count,
        2000
    )

    numeric_token_count = min(
        numeric_token_count,
        1000
    )

    return [

        message_length,
        token_count,
        digit_ratio,
        uppercase_ratio,
        special_char_count,
        numeric_token_count,
        unique_token_ratio,

        contains_ip,
        contains_url,
        contains_path,
        contains_port,
        contains_hex,

        error_keyword_count,

        contains_error,
        contains_warning,
        contains_timeout,

        contains_failure,
        contains_exception,

        repeated_token_ratio,
        average_token_length
    ]

structured = np.array([
    extract_structured_features(t)
    for t in X_text
])

print("\nStructured Feature Shape:")

print(structured.shape)

# ==============================
# 12. STRUCTURED FEATURE ANALYSIS
# ==============================

structured_df = pd.DataFrame(
    structured,
    columns=[

        "message_length",
        "token_count",
        "digit_ratio",
        "uppercase_ratio",
        "special_char_count",
        "numeric_token_count",
        "unique_token_ratio",

        "contains_ip",
        "contains_url",
        "contains_path",
        "contains_port",
        "contains_hex",

        "error_keyword_count",

        "contains_error",
        "contains_warning",
        "contains_timeout",

        "contains_failure",
        "contains_exception",

        "repeated_token_ratio",
        "average_token_length"
    ]
)

print("\nStructured Feature Sample:")

print(
    structured_df.head()
)

print("\nStructured Feature Statistics:")

print(
    structured_df.describe()
)

print("\nMissing Values:")

print(
    structured_df
    .isnull()
    .sum()
)

# ==============================
# 13. SCALE STRUCTURED FEATURES
# ==============================

scaler = StandardScaler()

structured_scaled = (
    scaler.fit_transform(structured)
)

print(
    "\nStructured Features Scaled Successfully"
)

# ==============================
# 14. COMBINE FEATURES
# ==============================

structured_sparse = (
    sparse.csr_matrix(
        structured_scaled
    )
)

X = hstack([
    X_tfidf,
    structured_sparse
])

print("\nFinal Feature Shape:")

print(X.shape)

# ==============================
# 15. SAVE OUTPUTS
# ==============================

os.makedirs(
    "E:/LogAnomalyDetector/models",
    exist_ok=True
)

os.makedirs(
    "E:/LogAnomalyDetector/data/processed",
    exist_ok=True
)

sparse.save_npz(
    "E:/LogAnomalyDetector/data/processed/X_features.npz",
    X
)

pd.DataFrame({

    "true_label": y,

    "source_dataset": (
        df["source_dataset"]
    )

}).to_csv(
    "E:/LogAnomalyDetector/data/processed/y_labels.csv",
    index=False
)

joblib.dump(
    vectorizer,
    "E:/LogAnomalyDetector/models/tfidf_vectorizer.joblib"
)

joblib.dump(
    scaler,
    "E:/LogAnomalyDetector/models/structured_scaler.joblib"
)

print(
    "\n✅ Final Advanced Adaptive Hybrid Multi-Dataset Feature Pipeline Complete"
)

HDFS Shape: (2000, 9)
HDFS Labels Shape: (575061, 2)
BGL Shape: (2000, 13)

HDFS Label Columns:
['BlockId', 'Label']

HDFS Label Sample:
                    BlockId    Label
0  blk_-1608999687919862906   Normal
1   blk_7503483334202473044   Normal
2  blk_-3544583377289625738  Anomaly
3  blk_-9073992586687739851   Normal
4   blk_7854771516489510256   Normal

Extracted Block IDs:
0       blk_38865049064139660
1    blk_-6952295868487656571
2     blk_7128370237687728475
3     blk_8229193803249955061
4    blk_-6670958622368987959
Name: BlockId, dtype: object

HDFS Label Distribution:
true_label
0    1931
1      69
Name: count, dtype: int64

HDFS Normalized Shape:
(2000, 3)

BGL Normalized Shape:
(2000, 3)

Combined Dataset Shape:
(4000, 3)

Dataset Distribution:
source_dataset
hdfs    2000
bgl     2000
Name: count, dtype: int64

After Cleaning:
(4000, 3)

Detected Vector-Like Rows: 0

Dataset Shape After Filtering: (4000, 3)

Label Distribution:
true_label
0    2074
1    1926
Name: count, d